In [0]:
%sql
CREATE TABLE IF NOT EXISTS mba.trusted.d_usinas(
    id_usina BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Chave substituta da dimensão usina',
    CodCEG STRING COMMENT 'Código do empreendimento de geração, sem a versão do CEG',
    NomEmpreendimento STRING COMMENT 'Nome do empreendimento de geração',
    DscOrigemCombustivel STRING COMMENT 'Origem do combustível utilizado pela usina',
    DscFonteCombustivel STRING COMMENT 'Descrição da fonte de combustível',
    DscTipoOutorga STRING COMMENT 'Tipo de outorga do empreendimento',
    NomFonteCombustivel STRING COMMENT 'Nome da fonte ou combustível utilizado',
    Latitude STRING COMMENT 'Latitude aproximada do empreendimento',
    Longitude STRING COMMENT 'Longitude aproximada do empreendimento',
    DscMuninicpios STRING COMMENT 'Municípios e estados onde o empreendimento está localizado',
    SigUFPrincipal STRING COMMENT 'UF principal do empreendimento',
    Potencia DECIMAL(20,2)COMMENT 'Potência outorgada do empreendimento em kW'
)
USING DELTA
COMMENT 'Dimensão de usinas e empreendimentos de geração de energia elétrica'

In [0]:
%sql
MERGE INTO mba.trusted.d_usinas AS tgt
USING
(
    SELECT DISTINCT substring_index( TRIM(CodCEG), '-', 1 ) AS CodCEG,
        TRIM(NomEmpreendimento) AS NomEmpreendimento,
        DscOrigemCombustivel,
        DscFonteCombustivel,
        DscTipoOutorga,
        NomFonteCombustivel,
        NumCoordNEmpreendimento AS Latitude,
        NumCoordEEmpreendimento AS Longitude,
        DscMuninicpios,
        SigUFPrincipal,
        CAST(MdaPotenciaOutorgadaKw AS DECIMAL(20,2)) AS Potencia
    FROM mba.raw.usinas
    WHERE CodCEG IS NOT NULL
) AS src
ON tgt.CodCEG = src.CodCEG

WHEN NOT MATCHED THEN
    INSERT ( CodCEG, NomEmpreendimento, DscOrigemCombustivel, DscFonteCombustivel,
        DscTipoOutorga, NomFonteCombustivel, Latitude, Longitude, DscMuninicpios, SigUFPrincipal, Potencia )

    VALUES( src.CodCEG, src.NomEmpreendimento, src.DscOrigemCombustivel, src.DscFonteCombustivel, src.DscTipoOutorga,
        src.NomFonteCombustivel, src.Latitude, src.Longitude, src.DscMuninicpios, src.SigUFPrincipal, src.Potencia );

In [0]:
dbutils.notebook.exit("Executed")